In [1]:
###### The following is the task of this logic

## 1. Pull all the 47 records from the google drive or from the downloaded file. 
## 2. Create a function that can take the ID and pull the 'References' & 'Citations' from OpenAlex. 
## 3. Add a parameter field that will filter out the papers that are below that threshold. 
## 4. Pull all the references & citations for the 47 odd papers. 
## 5. Filter out the once that do not meet similarity threshold.  
## 6. In the filtered out list search it through the word cloud. And if any matches then add it to the combined pool. 
## 6. In the pooled list see if there are any new papers that have to be passed through API. 
## 7. Pull the reference & citation related from API info. 
## 8. Add to the pool if they do not exists in the list & meet the threshold & word cloud tests. 



In [39]:
# Multiple ways to get and work with home directory
import os
from pathlib import Path
import spacy
import scispacy
import pandas as pd
import math
import numpy as np
import requests
import re
import time
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from sentence_transformers import SentenceTransformer, util
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine_similarity
from torch.nn.functional import cosine_similarity as torch_cosine_similarity
from functools import lru_cache
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

In [37]:
# 1. Using os.path
home_dir = os.path.expanduser('~')
print(f"Home directory (os.path): {home_dir}")

# 2. Using Path from pathlib (modern approach)
home = Path.home()
print(f"Home directory (pathlib): {home}")

# 3. Using environment variables
home_env = os.environ['HOME']  # For Unix/Linux/Mac
# home_env = os.environ['USERPROFILE']  # For Windows
print(f"Home directory (environ): {home_env}")

# Example: List contents of home directory
#print("\nContents of home directory:")
#for item in Path.home().iterdir():
#    print(f"- {item.name}")

# Create a new folder (if it doesn't exist)
new_folder = "/Users/naveendeevi/Datasets"
if not os.path.exists(new_folder):
    os.makedirs(new_folder)

# Change directory to the new folder
os.chdir(new_folder)

# Verify current working directory
print(f"Current working directory: {os.getcwd()}")

Home directory (os.path): /Users/naveendeevi
Home directory (pathlib): /Users/naveendeevi
Home directory (environ): /Users/naveendeevi
Current working directory: /Users/naveendeevi/Datasets


### 1. Create a function for pulling the Citations & References from the OpenAlex API. 

In [97]:
def get_openalex_references_and_citations(work_id):
    """
    Given an OpenAlex Work ID, return a dictionary of its references and citations.

    Args:
        work_id (str): OpenAlex Work ID (e.g., "W4406696697")

    Returns:
        dict: {
            'id': str,
            'title': str,
            'referenced_works': List[str],
            'cited_by_works': List[str]
        }
    """
    base_url = f"https://api.openalex.org/works/{work_id}"
    response = requests.get(base_url)

    if response.status_code != 200:
        raise Exception(f"Error fetching data for {work_id}: {response.status_code}")

    data = response.json()
    referenced_works = data.get("referenced_works", [])
    
    # Get citing works via filter
    citations_url = f"https://api.openalex.org/works?filter=cites:{work_id}&per-page=200"
    citations_response = requests.get(citations_url)

    if citations_response.status_code != 200:
        raise Exception(f"Error fetching citations for {work_id}: {citations_response.status_code}")
    
    citations_data = citations_response.json()
    citing_ids = [item["id"].split("/")[-1] for item in citations_data.get("results", [])]

    return {
        "id": data.get("id", "").split("/")[-1],
        "title": data.get("title", ""),
        "referenced_works": referenced_works,
        "cited_by_works": citing_ids
    }


## Function that populates the Pool Collection DF. 
def create_openalex_entry_df(identifier):
    """
    Given an OpenAlex ID or link, return a DataFrame with metadata:
    ID, Title, API Link, Citation/Reference, and Author Names.

    Args:
        identifier (str): OpenAlex Work ID (e.g., 'W2046695653') or URL (e.g., 'https://openalex.org/W2049189688')

    Returns:
        pd.DataFrame: With columns ['ID', 'Title', 'API link', 'Citation or Reference', 'Authors']
    """
    # Determine if input is a link or ID
    if identifier.startswith("https://openalex.org/"):
        work_id = identifier.split("/")[-1]
        label = "Reference"
    else:
        work_id = identifier
        label = "Citation"

    # Build API link
    api_link = f"https://api.openalex.org/works/{work_id}"

    # Fetch paper metadata
    response = requests.get(api_link)
    if response.status_code != 200:
        raise Exception(f"Failed to retrieve OpenAlex data: {response.status_code}")

    data = response.json()

    title = data.get("title", "")
    authorships = data.get("authorships", [])

    # Extract author names
    author_names = [auth["author"]["display_name"] for auth in authorships if "author" in auth]
    authors_str = ", ".join(author_names) if author_names else ""

    # Assemble row
    row = {
        "ID": work_id,
        "Title": title,
        "API link": api_link,
        "Citation or Reference": label,
        "Authors": authors_str
    }

    return pd.DataFrame([row])

## Similarity scores SentenceBERT

# Corpus-wide SBERT Similarity (One-to-Many)
def sbert_group_similarity(df, ref_col, cmp_col, score_col='SBERT_group_similarity'):
    model = SentenceTransformer('all-MiniLM-L6-v2')

    df = df.copy()
    df['ref_group'] = (df[ref_col] != df[ref_col].shift()).cumsum()

    all_scores = {}

    for gid, group in df.groupby('ref_group'):
        ref_text = group[ref_col].iloc[0]

        # Skip entire group if reference text is missing or invalid
        if pd.isna(ref_text) or not isinstance(ref_text, str) or not ref_text.strip():
            for idx in group.index:
                all_scores[idx] = np.nan
            continue

        cmp_texts = group[cmp_col].tolist()
        cmp_indices = group.index.tolist()

        sim_scores = []
        for t in cmp_texts:
            if pd.isna(t) or not isinstance(t, str) or not t.strip():
                sim_scores.append(np.nan)
            else:
                emb_ref = model.encode(ref_text, convert_to_tensor=True)
                emb_cmp = model.encode(t, convert_to_tensor=True)
                score = util.cos_sim(emb_ref, emb_cmp).item()
                sim_scores.append(score)

        for i, idx in enumerate(cmp_indices):
            all_scores[idx] = sim_scores[i]

    df[score_col] = df.index.to_series().map(all_scores)
    return df.drop(columns=['ref_group'])

# Word cloud & Sentence similarity
def word_cloud_jaccard_similarity(word_list, sentence):
    """
    Compute Jaccard similarity between a list of words and a sentence.

    Args:
        word_list (list): A list of words (e.g., MDD_Word_Cloud)
        sentence (str): A sentence to compare with the word list

    Returns:
        float: Jaccard similarity score (0.0 to 1.0)
    """
    if not isinstance(sentence, str) or not sentence.strip():
        return 0.0

    # Convert both to sets
    set1 = set(w.lower() for w in word_list)
    set2 = set(sentence.lower().split())

    # Compute Jaccard index
    intersection = set1.intersection(set2)
    union = set1.union(set2)

    if not union:
        return 0.0

    return len(intersection) / len(union)

MDD_Word_Cloud = ["5-HT1A", "5-HT4R", "Agomelatine", "AMPA Receptor Activation", "AMPA antagonist", "Analgesics", "Anesthetics",
    "Antidepressant", "Antidepressants", "Antipsychotic", "Antipsychotics", "Ayahuasca", "BDI", "BDNF", "Beta-arrestin",
    "Biomarker", "Biomarkers", "Brexpiprazole", "Bupropion", "Citalopram", "Clomipramine", "CYP2C19", "CYP2D6",
    "Central Nervous System Depressants", "Cyclooxygenase Inhibitors", "Desvenlafaxine", "Dopamine Reuptake Inhibitor",
    "Depression", "Depress", "Depressive", "MDD", "Duloxetine", "EEG", "Escitalopram", "Estrogen Antagonists", 
    "Excitatory Amino Acid Agents", "Excitatory Amino Acid Antagonists",
    "Fluoxetine", "GABA Agents", "Gene", "GeneSight", "Genetic", "Genetics", "Genomics", "General Anesthetics",
    "Glutamate", "Glutamate antagonist", "HAMD", "HAM-D", "HDRS", "Hydroxynorketamine", "IGF-1", "Inventory (generic)",
    "Ketamine", "Leucine", "MADRS", "MDMA", "MTHFR", "Magnetic Resonance Spectroscopy", "Magnetoencephalography",
    "Metabotropic glutamate receptor antagonist", "Mirtazapine", "Montgomery-Åsberg Depression Rating Scale",
    "NEGR1", "NMDA antagonist", "NMDA receptor antagonist", "Norepinephrine Reuptake Inhibitor", "Opioid",
    "PACAP", "PHQ", "PHQ-8", "PHQ-9", "PRKCB1", "Paroxetine", "Patient Health Questionnaire", "Peripheral Nervous System Agents",
    "Psilocybin", "P2X7", "QEEG", "QIDS", "SERT", "SNRI", "SNRIs", "SSRI", "SSRIs", "SV2A", "Selective Serotonin Reuptake Inhibitor",
    "Sensory System Agents", "Serotonin Uptake Inhibitors", "Serotonin transporter gene polymorphism",
    "SPECT", "Trazodone", "Thyroid pathway genes", "Triiodothyronine", "Venlafaxine", "Vilazodone", "Vortioxetine"
]

# Create a function that takes a dataframe and give a Approved & Rejected dataframe

def take_pool_df_approved_rejected_df(pool_collection,similarity_threshold=0.90): 
    
    ###################################2. Pull citations & References from seed sheets #####################################
    for ids,title in zip(pool_collection['ID'],pool_collection['Title']):
        citations_references = get_openalex_references_and_citations(ids)
        #print(ids)
        filter_sheet = pd.DataFrame(columns=["seed_title","reference_ID","reference_citation_title","API link","Authors","SBERT_score","word_cloud_score"])
        #print('Seed title:%s'%title)
        #filter_sheet.loc[len(filter_sheet),'seed_title'] = title
        # Pull references
        for i in citations_references['referenced_works']:
            x = create_openalex_entry_df(i)
            #print('Title: %s'%x['Title'])
            #Sub_pool_collection = pd.concat([Sub_pool_collection,x])
            filter_sheet.loc[len(filter_sheet),['seed_title','reference_ID','reference_citation_title','API link','Authors']] = [title,x.loc[0,'ID'],x.loc[0,'Title'],x.loc[0,'API link'],x.loc[0,'Authors']]
        
        for j in citations_references['cited_by_works']:
            y = create_openalex_entry_df(j)
            #Sub_pool_collection = pd.concat([Sub_pool_collection,y])
            #filter_sheet.loc[len(filter_sheet),['seed_title','reference_citation_title']] = [title,y.loc[0,'Title']]
            filter_sheet.loc[len(filter_sheet),['seed_title','reference_ID','reference_citation_title','API link','Authors']] = [title,y.loc[0,'ID'],y.loc[0,'Title'],y.loc[0,'API link'],y.loc[0,'Authors']]

        ####################### 3. Add filtering criteria. a) SentenceBERT b)Word Cloud ##########################################
        filter_sheet_V2 = sbert_group_similarity(filter_sheet,'seed_title','reference_citation_title', score_col = 'SBERT_score')

        # Min-Max Normalization: scale values to range [0, 1]
        min_val = filter_sheet_V2['SBERT_score'].min()
        max_val = filter_sheet_V2['SBERT_score'].max()
        filter_sheet_V2['SBERT_score_normalized'] = (filter_sheet_V2['SBERT_score'] - min_val) / (max_val - min_val)
        filter_sheet_V2.drop(columns=['SBERT_score'],inplace=True)

        filter_sheet_V2['word_cloud_score'] = filter_sheet_V2['reference_citation_title'].apply(lambda sentence: word_cloud_jaccard_similarity(MDD_Word_Cloud, sentence))

        # Filter for the papers that are above the similarity threshold and word cloud. 
        filter_sheet_V2['title_Status'] = np.where((filter_sheet_V2['SBERT_score_normalized'] >= similarity_threshold) | (filter_sheet_V2['word_cloud_score'] > 0.00), 'collection_pool', 'rejection_pool')

    # === APPROVED POOL --> Pooled_collection ===
    approved_df = filter_sheet_V2[filter_sheet_V2['title_Status'] == 'collection_pool']
    approved_df = approved_df[['seed_title','reference_ID', 'reference_citation_title', 'API link', 'Authors']].copy()
    approved_df.columns = ['seed_title','Linked_Paper_ID', 'Linked_Paper_Title', 'Linked Paper API link', 'Linked Paper Authors']
    #approved_df['Citation or Reference'] = 'reference_&_citations'

    # === REJECTED POOL --> Rejection_pool ===
    rejected_df = filter_sheet_V2[filter_sheet_V2['title_Status'] == 'rejection_pool']
    rejected_df = rejected_df[['reference_ID', 'reference_citation_title', 'API link', 'Authors']].copy()
    rejected_df.columns = ['Linked_Paper_ID', 'Linked_Paper_Title', 'Linked Paper API link', 'Linked Paper Authors']
    #rejected_df['Citation or Reference'] = 'reference_&_citations'

    return approved_df,rejected_df


In [5]:
x = create_openalex_entry_df('W2120826795')

In [6]:
x

,ID,Title,API link,Citation or Reference,Authors
0,W2120826795,"Depression, anxiety and telomere length in you...",https://api.openalex.org/works/W2120826795,Citation,"Belinda L. Needham, Briana Mezuk, Natalie Bare..."


In [7]:
pool_collection = pd.DataFrame(columns=["ID", "Title", "API link", "Citation or Reference"])

In [8]:
pool_collection = pd.concat([pool_collection,x])

In [72]:
similarity_threshold = 0.10

start_time = time.time()


###########################1. Read the papers that Adam & Eleni collected. That is the seed paper. Add it to the seed paper ###############
seed_papers = pd.read_csv('PHQ9_publication_index_full_titles_authors.csv')

seed_list = ['W4394693583','W1528826381']

#seed_list = ['W2148132316','W2114058973','W2148132316','W2114058973','W2153644230','W2742814779','W2338977821','W2014381681','W2012109041','W2140505555',
#             'W2775576435','W2257045848']

pool_collection = pd.DataFrame(columns=["ID", "Title", "API link", "Authors", "Citation or Reference"])


for i in seed_list:
    pool_collection = pd.concat([pool_collection,create_openalex_entry_df(i)], ignore_index = True)
    
pool_collection['Citation or Reference'] = 'Seed' # As these are the seed papers



###################################2. Pull citations & References from seed sheets #####################################
for ids,title in zip(pool_collection['ID'],pool_collection['Title']):
    citations_references = get_openalex_references_and_citations(ids)
    #print(ids)
    filter_sheet = pd.DataFrame(columns=["seed_title","reference_ID","reference_citation_title","API link","Authors","SBERT_score","word_cloud_score"])
    #print('Seed title:%s'%title)
    #filter_sheet.loc[len(filter_sheet),'seed_title'] = title
    # Pull references
    for i in citations_references['referenced_works']:
        x = create_openalex_entry_df(i)
        #print('Title: %s'%x['Title'])
        #Sub_pool_collection = pd.concat([Sub_pool_collection,x])
        filter_sheet.loc[len(filter_sheet),['seed_title','reference_ID','reference_citation_title','API link','Authors']] = [title,x.loc[0,'ID'],x.loc[0,'Title'],x.loc[0,'API link'],x.loc[0,'Authors']]
    
    for j in citations_references['cited_by_works']:
        y = create_openalex_entry_df(j)
        #Sub_pool_collection = pd.concat([Sub_pool_collection,y])
        #filter_sheet.loc[len(filter_sheet),['seed_title','reference_citation_title']] = [title,y.loc[0,'Title']]
        filter_sheet.loc[len(filter_sheet),['seed_title','reference_ID','reference_citation_title','API link','Authors']] = [title,y.loc[0,'ID'],y.loc[0,'Title'],y.loc[0,'API link'],y.loc[0,'Authors']]



####################### 3. Add filtering criteria. a) SentenceBERT b)Word Cloud ##########################################
    filter_sheet_V2 = sbert_group_similarity(filter_sheet,'seed_title','reference_citation_title', score_col = 'SBERT_score')

# Min-Max Normalization: scale values to range [0, 1]
    min_val = filter_sheet_V2['SBERT_score'].min()
    max_val = filter_sheet_V2['SBERT_score'].max()
    filter_sheet_V2['SBERT_score_normalized'] = (filter_sheet_V2['SBERT_score'] - min_val) / (max_val - min_val)
    filter_sheet_V2.drop(columns=['SBERT_score'],inplace=True)

    filter_sheet_V2['word_cloud_score'] = filter_sheet_V2['reference_citation_title'].apply(lambda sentence: word_cloud_jaccard_similarity(MDD_Word_Cloud, sentence))

# Filter for the papers that are above the similarity threshold and word cloud. 
    similarity_threshold = 0.10
    filter_sheet_V2['title_Status'] = np.where((filter_sheet_V2['SBERT_score_normalized'] >= similarity_threshold) | (filter_sheet_V2['word_cloud_score'] > 0.00), 'collection_pool', 'rejection_pool')



####################### 4. Add new papers to Pooled_collection & rejected new papers to Rejection_pool#############################
# === APPROVED POOL --> Pooled_collection ===
    approved_df = filter_sheet_V2[filter_sheet_V2['title_Status'] == 'collection_pool']
    approved_df = approved_df[['reference_ID', 'reference_citation_title', 'API link', 'Authors']].copy()
    approved_df.columns = ['ID', 'Title', 'API link', 'Authors']
    approved_df['Citation or Reference'] = 'reference_&_citations'

# Add only new approved entries
    existing_ids = set(pool_collection['ID'])
    new_approved_entries = approved_df[~approved_df['ID'].isin(existing_ids)]

    new_approved_count = len(new_approved_entries)
    print(f"New entries to be added to pool_collection: {new_approved_count}")

    if new_approved_count > 0:
        pool_collection = pd.concat([pool_collection, new_approved_entries], ignore_index=True)

# === REJECTED POOL --> Rejection_pool ===
    rejected_df = filter_sheet_V2[filter_sheet_V2['title_Status'] == 'rejection_pool']
    rejected_df = rejected_df[['reference_ID', 'reference_citation_title', 'API link', 'Authors']].copy()
    rejected_df.columns = ['ID', 'Title', 'API link', 'Authors']
    rejected_df['Citation or Reference'] = 'reference_&_citations'

# Create rejected_pool if not already created
    try:
        rejected_pool
    except NameError:
        rejected_pool = pd.DataFrame(columns=['ID', 'Title', 'API link', 'Authors', 'Citation or Reference'])

# Add only new rejected entries
    existing_rejected_ids = set(rejected_pool['ID'])
    new_rejected_entries = rejected_df[~rejected_df['ID'].isin(existing_rejected_ids)]

    new_rejected_count = len(new_rejected_entries)
    print(f"New entries to be added to rejected_pool: {new_rejected_count}")

    if new_rejected_count > 0:
        rejected_pool = pd.concat([rejected_pool, new_rejected_entries], ignore_index=True)

    print('Completed for: %s, %s'%(ids,title))


end_time = time.time()
elapsed_time = end_time - start_time

print(f"Execution time: {elapsed_time:.2f} seconds")


/opt/anaconda3/envs/TransformersCleanNLPEnv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


New entries to be added to pool_collection: 56
New entries to be added to rejected_pool: 0
Completed for: W4394693583, Acute Adverse Effects of Therapeutic Doses of Psilocybin


/opt/anaconda3/envs/TransformersCleanNLPEnv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


New entries to be added to pool_collection: 98
New entries to be added to rejected_pool: 3
Completed for: W1528826381, Monoamine oxidase inhibitors (MAOIs) for fibromyalgia syndrome
Execution time: 152.89 seconds


In [95]:


###########################1. Read the papers that Adam & Eleni collected. That is the seed paper. Add it to the seed paper ###############
seed_papers = pd.read_csv('PHQ9_publication_index_full_titles_authors.csv')

seed_list = ['W4394693583','W1528826381']

#seed_list = ['W2148132316','W2114058973','W2148132316','W2114058973','W2153644230','W2742814779','W2338977821','W2014381681','W2012109041','W2140505555',
#             'W2775576435','W2257045848']

pool_collection = pd.DataFrame(columns=["Seed_ID", "Seed_Title", "Seed API link", "Seed Authors", "Seed Citation or Reference"])


for i in seed_list:
    pool_collection = pd.concat([pool_collection,create_openalex_entry_df(i)], ignore_index = True)
    
pool_collection['Citation or Reference'] = 'Seed' # As these are the seed papers

########################### 2. Execute the logic that passes pool_collection and returns Approved list & Rejected list ###############

approved_list, rejected_list = take_pool_df_approved_rejected_df(pool_collection)
#combined_df = pd.concat([pool_collection, approved_list], axis=1)





/opt/anaconda3/envs/TransformersCleanNLPEnv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/opt/anaconda3/envs/TransformersCleanNLPEnv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [96]:
pool_collection

,Seed_ID,Seed_Title,Seed API link,Seed Authors,Seed Citation or Reference,ID,Title,API link,Citation or Reference,Authors
0,NaN,NaN,NaN,NaN,NaN,W4394693583,Acute Adverse Effects of Therapeutic Doses of ...,https://api.openalex.org/works/W4394693583,Seed,"Akhila Yerubandi, Jennifer E. Thomas, N. M. Ma..."
1,NaN,NaN,NaN,NaN,NaN,W1528826381,Monoamine oxidase inhibitors (MAOIs) for fibro...,https://api.openalex.org/works/W1528826381,Seed,"Sera Tort, Gerard Urrútia, María Betina Nishis..."


In [92]:
approved_list['seed_title'].unique()

array(['Monoamine oxidase inhibitors (MAOIs) for fibromyalgia syndrome'],
      dtype=object)

In [88]:
combined_df = pd.concat([pool_collection, approved_list], axis=1)

In [89]:
combined_df.head(3)

,Seed_ID,Seed_Title,Seed API link,Seed Authors,Seed Citation or Reference,ID,Title,API link,Citation or Reference,Authors,seed_title,Linked_Paper_ID,Linked_Paper_Title,Linked Paper API link,Linked Paper Authors
0,NaN,NaN,NaN,NaN,NaN,W4394693583,Acute Adverse Effects of Therapeutic Doses of ...,https://api.openalex.org/works/W4394693583,Seed,"Akhila Yerubandi, Jennifer E. Thomas, N. M. Ma...",Monoamine oxidase inhibitors (MAOIs) for fibro...,W119219963,Primary fibromyalgia syndrome: current concepts.,https://api.openalex.org/works/W119219963,M. B. Yunus
1,NaN,NaN,NaN,NaN,NaN,W1528826381,Monoamine oxidase inhibitors (MAOIs) for fibro...,https://api.openalex.org/works/W1528826381,Seed,"Sera Tort, Gerard Urrútia, María Betina Nishis...",Monoamine oxidase inhibitors (MAOIs) for fibro...,W155475857,Pharmacology of moclobemide.,https://api.openalex.org/works/W155475857,"W. Haefely, W.P. Burkard, Andrea M. Cesura, A...."
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Monoamine oxidase inhibitors (MAOIs) for fibro...,W1568215891,Anti-depressants and centrally active agents f...,https://api.openalex.org/works/W1568215891,"María Betina Nishishinya, Brian Walitt, Gerard..."


In [80]:
approved_list.head(3)

,ID,Title,API link,Authors,Citation or Reference
0,W119219963,Primary fibromyalgia syndrome: current concepts.,https://api.openalex.org/works/W119219963,M. B. Yunus,reference_&_citations
1,W155475857,Pharmacology of moclobemide.,https://api.openalex.org/works/W155475857,"W. Haefely, W.P. Burkard, Andrea M. Cesura, A....",reference_&_citations
2,W1568215891,Anti-depressants and centrally active agents f...,https://api.openalex.org/works/W1568215891,"María Betina Nishishinya, Brian Walitt, Gerard...",reference_&_citations


In [81]:
rejected_list.head(3)

,ID,Title,API link,Authors,Citation or Reference
46,W2470711321,Deleted Work,https://api.openalex.org/works/W2470711321,,reference_&_citations
51,W4285719527,Deleted Work,https://api.openalex.org/works/W4285719527,,reference_&_citations
96,W2490019368,Untersuchung arbeitsbezogener Endpunkte in ran...,https://api.openalex.org/works/W2490019368,Ingmar Wolf,reference_&_citations


In [74]:
pool_collection

,ID,Title,API link,Authors,Citation or Reference
0,W4394693583,Acute Adverse Effects of Therapeutic Doses of ...,https://api.openalex.org/works/W4394693583,"Akhila Yerubandi, Jennifer E. Thomas, N. M. Ma...",Seed
1,W1528826381,Monoamine oxidase inhibitors (MAOIs) for fibro...,https://api.openalex.org/works/W1528826381,"Sera Tort, Gerard Urrútia, María Betina Nishis...",Seed
2,W2006949123,Possible involvement of the central dopaminerg...,https://api.openalex.org/works/W2006949123,"M.K. Menon, W. G. Clark, David T. Masuoka",reference_&_citations
3,W2018241451,Identification of serotonin 5-HT1A receptor pa...,https://api.openalex.org/works/W2018241451,"Andreas Nievergelt, Peter Huonker, Roland Scho...",reference_&_citations
4,W2022443784,"Psilocybin dose-dependently causes delayed, tr...",https://api.openalex.org/works/W2022443784,"Matthew W. Johnson, R. Andrew Sewell, Roland R...",reference_&_citations
...,...,...,...,...,...
151,W2405785408,Efectos terapéuticos del yoga en la fibromialg...,https://api.openalex.org/works/W2405785408,"Vanesa Náger Obón, Miguel Ángel Oviedo, Javier...",reference_&_citations
152,W4245869676,Fibromyalgia,https://api.openalex.org/works/W4245869676,Levan Atanelov,reference_&_citations
153,W4319229480,Phenelzine-based probes reveal Secernin-3 is i...,https://api.openalex.org/works/W4319229480,"Katelyn A. Bustin, Kyosuke Shishikura, Irene C...",reference_&_citations
154,W2990987883,Current drug treatment in fibromyalgia,https://api.openalex.org/works/W2990987883,Seval Pehlevan,reference_&_citations


In [41]:
import pandas as pd
import numpy as np
import time
from functools import lru_cache
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
import requests

# ====== CONFIG ======
seed_list = ['W2148132316']  # manually provided seed list
similarity_threshold = 0.90
MAX_THREADS = 3
RETRY_DELAY = 2
MAX_RETRIES = 3

# ====== RETRYABLE API CALL WRAPPER ======
def create_openalex_entry_df_with_retry(work_id, retries=MAX_RETRIES, delay=RETRY_DELAY):
    for attempt in range(retries):
        try:
            return create_openalex_entry_df(work_id)
        except Exception as e:
            if '429' in str(e) and attempt < retries - 1:
                print(f"⚠️ Rate limit hit for {work_id}, retrying in {delay} seconds...")
                time.sleep(delay)
            else:
                raise

# ====== CACHED ENTRY FETCHER WITH THROTTLING ======
@lru_cache(maxsize=None)
def cached_create_openalex_entry_df(work_id):
    time.sleep(0.5)  # Throttle to avoid API rate limit
    return create_openalex_entry_df_with_retry(work_id)

# ====== PARALLEL FETCH WRAPPER ======
def get_entries_parallel(ids):
    with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
        results = list(executor.map(cached_create_openalex_entry_df, ids))
    return results

# ====== INITIALIZE POOLS ======
pool_collection = pd.DataFrame(columns=["ID", "Title", "API link", "Authors", "Citation or Reference"])
rejected_pool = pd.DataFrame(columns=["ID", "Title", "API link", "Authors", "Citation or Reference"])

# ====== ADD SEED PAPERS TO POOL ======
for seed_id in seed_list:
    entry = cached_create_openalex_entry_df(seed_id)
    entry['Citation or Reference'] = 'Seed'
    pool_collection = pd.concat([pool_collection, entry], ignore_index=True)

# ====== EXPANSION LOOP ======
start_time = time.time()
processed_ids = set()
current_index = 0

while current_index < len(pool_collection):
    seed_row = pool_collection.iloc[current_index]
    seed_id, seed_title = seed_row['ID'], seed_row['Title']

    if seed_id in processed_ids:
        current_index += 1
        continue

    print(f"\n\u25b6\ufe0f Processing: {seed_id} — {seed_title}")
    processed_ids.add(seed_id)

    # Step 1: Fetch citations + references
    citations_references = get_openalex_references_and_citations(seed_id)
    combined_ids = citations_references['referenced_works'] + citations_references['cited_by_works']
    combined_ids = list(set(combined_ids))  # deduplicate

    entries = get_entries_parallel(combined_ids)

    filter_rows = []
    for entry in entries:
        if not entry.empty:
            filter_rows.append([
                seed_title,
                entry.loc[0, 'ID'],
                entry.loc[0, 'Title'],
                entry.loc[0, 'API link'],
                entry.loc[0, 'Authors']
            ])

    filter_sheet = pd.DataFrame(filter_rows, columns=["seed_title", "reference_ID", "reference_citation_title", "API link", "Authors"])

    # Step 2: Apply similarity filters
    filter_sheet_V2 = sbert_group_similarity(filter_sheet, 'seed_title', 'reference_citation_title', score_col='SBERT_score')
    min_val, max_val = filter_sheet_V2['SBERT_score'].min(), filter_sheet_V2['SBERT_score'].max()
    filter_sheet_V2['SBERT_score_normalized'] = (filter_sheet_V2['SBERT_score'] - min_val) / (max_val - min_val)
    filter_sheet_V2.drop(columns=['SBERT_score'], inplace=True)
    filter_sheet_V2['word_cloud_score'] = filter_sheet_V2['reference_citation_title'].apply(lambda sentence: word_cloud_jaccard_similarity(MDD_Word_Cloud, sentence))
    filter_sheet_V2['title_Status'] = np.where(
        (filter_sheet_V2['SBERT_score_normalized'] >= similarity_threshold) |
        (filter_sheet_V2['word_cloud_score'] > 0.00),
        'collection_pool',
        'rejection_pool'
    )

    # Step 3: Add to pools
    existing_ids = set(pool_collection['ID'])
    new_pool_rows, new_reject_rows = [], []

    for _, row in filter_sheet_V2.iterrows():
        new_entry = {
            'ID': row['reference_ID'],
            'Title': row['reference_citation_title'],
            'API link': row['API link'],
            'Authors': row['Authors'],
            'Citation or Reference': 'reference_&_citations'
        }
        if row['reference_ID'] in existing_ids:
            continue

        if row['title_Status'] == 'collection_pool':
            new_pool_rows.append(new_entry)
        else:
            if row['reference_ID'] not in set(rejected_pool['ID']):
                new_reject_rows.append(new_entry)

    print(f"  ✅ New approved: {len(new_pool_rows)} | ❌ Rejected: {len(new_reject_rows)}")

    if new_pool_rows:
        pool_collection = pd.concat([pool_collection, pd.DataFrame(new_pool_rows)], ignore_index=True)
    if new_reject_rows:
        rejected_pool = pd.concat([rejected_pool, pd.DataFrame(new_reject_rows)], ignore_index=True)

    current_index += 1

# ====== END REPORT ======
end_time = time.time()
print(f"\n\ud83c\udf1f Total papers in pool_collection: {len(pool_collection)}")
print(f"\ud83d\uddd1\ufe0f Total papers in rejected_pool: {len(rejected_pool)}")
print(f"\u23f3 Execution time: {end_time - start_time:.2f} seconds")



▶️ Processing: W2148132316 — Evidence for a differential role of HPA-axis function, inflammation and metabolic syndrome in melancholic versus atypical depression


/opt/anaconda3/envs/TransformersCleanNLPEnv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


  ✅ New approved: 265 | ❌ Rejected: 4

▶️ Processing: W2025725983 — Inflammatory mechanisms in major depressive disorder


/opt/anaconda3/envs/TransformersCleanNLPEnv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


  ✅ New approved: 192 | ❌ Rejected: 4

▶️ Processing: W2034098487 — Genetics of recurrent early‐onset depression (GenRED): Design and preliminary clinical characteristics of a repository sample for genetic linkage studies


KeyboardInterrupt: 

In [60]:
import pandas as pd
import numpy as np
import time
from functools import lru_cache
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
import requests

# ====== CONFIG ======
seed_list = ['W4394693583']  # manually provided seed list
similarity_threshold = 0.90
MAX_THREADS = 3
RETRY_DELAY = 2
MAX_RETRIES = 3
MAX_HOPS = 1  # NEW: configurable number of hops

# ====== RETRYABLE API CALL WRAPPER ======
def create_openalex_entry_df_with_retry(work_id, retries=MAX_RETRIES, delay=RETRY_DELAY):
    for attempt in range(retries):
        try:
            return create_openalex_entry_df(work_id)
        except Exception as e:
            if ('429' in str(e) or 'SSLError' in str(e)) and attempt < retries - 1:
                print(f"⚠️ Temporary error for {work_id} ({e}), retrying in {delay} seconds...")
                time.sleep(delay)
            else:
                raise

# ====== CACHED ENTRY FETCHER WITH THROTTLING ======
@lru_cache(maxsize=None)
def cached_create_openalex_entry_df(work_id):
    time.sleep(0.5)  # Throttle to avoid API rate limit
    return create_openalex_entry_df_with_retry(work_id)

# ====== PARALLEL FETCH WRAPPER ======
def get_entries_parallel(ids):
    with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
        results = list(executor.map(cached_create_openalex_entry_df, ids))
    return results

# ====== INITIALIZE POOLS ======
pool_collection = pd.DataFrame(columns=["ID", "Title", "API link", "Authors", "Citation or Reference"])
rejected_pool = pd.DataFrame(columns=["ID", "Title", "API link", "Authors", "Citation or Reference"])

# ====== ADD SEED PAPERS TO POOL ======
for seed_id in seed_list:
    entry = cached_create_openalex_entry_df(seed_id)
    entry['Citation or Reference'] = 'Seed'
    pool_collection = pd.concat([pool_collection, entry], ignore_index=True)

# ====== EXPANSION WITH HOP CONTROL ======
start_time = time.time()

# Breadth-first traversal up to MAX_HOPS
processed_ids = set()
current_index = 0
current_hop = 0

while current_index < len(pool_collection) and current_hop < MAX_HOPS:
    seed_row = pool_collection.iloc[current_index]
    seed_id, seed_title = seed_row['ID'], seed_row['Title']

    if seed_id in processed_ids:
        current_index += 1
        continue

    print(f"Processing hop {current_hop + 1}: {seed_id} - {seed_title}")
    processed_ids.add(seed_id)

    # Step 1: Fetch citations + references
    citations_references = get_openalex_references_and_citations(seed_id)
    combined_ids = citations_references['referenced_works'] + citations_references['cited_by_works']
    combined_ids = list(set(combined_ids))

    entries = get_entries_parallel(combined_ids)

    filter_rows = []
    for entry in entries:
        if not entry.empty:
            filter_rows.append([
                seed_title,
                entry.loc[0, 'ID'],
                entry.loc[0, 'Title'],
                entry.loc[0, 'API link'],
                entry.loc[0, 'Authors']
            ])

    filter_sheet = pd.DataFrame(filter_rows, columns=["seed_title", "reference_ID", "reference_citation_title", "API link", "Authors"])

    # Step 2: Apply similarity filters
    filter_sheet_V2 = sbert_group_similarity(filter_sheet, 'seed_title', 'reference_citation_title', score_col='SBERT_score')
    min_val, max_val = filter_sheet_V2['SBERT_score'].min(), filter_sheet_V2['SBERT_score'].max()
    filter_sheet_V2['SBERT_score_normalized'] = (filter_sheet_V2['SBERT_score'] - min_val) / (max_val - min_val)
    filter_sheet_V2.drop(columns=['SBERT_score'], inplace=True)
    filter_sheet_V2['word_cloud_score'] = filter_sheet_V2['reference_citation_title'].apply(lambda sentence: word_cloud_jaccard_similarity(MDD_Word_Cloud, sentence))
    filter_sheet_V2['title_Status'] = np.where(
        (filter_sheet_V2['SBERT_score_normalized'] >= similarity_threshold) |
        (filter_sheet_V2['word_cloud_score'] > 0.00),
        'collection_pool',
        'rejection_pool'
    )

    # Step 3: Add to pools
    existing_ids = set(pool_collection['ID'])
    new_pool_rows, new_reject_rows = [], []

    for _, row in filter_sheet_V2.iterrows():
        new_entry = {
            'ID': row['reference_ID'],
            'Title': row['reference_citation_title'],
            'API link': row['API link'],
            'Authors': row['Authors'],
            'Citation or Reference': 'reference_&_citations'
        }
        if row['reference_ID'] in existing_ids:
            continue

        if row['title_Status'] == 'collection_pool':
            new_pool_rows.append(new_entry)
        else:
            if row['reference_ID'] not in set(rejected_pool['ID']):
                new_reject_rows.append(new_entry)

    print(f"✅ New approved: {len(new_pool_rows)} | ❌ Rejected: {len(new_reject_rows)}")

    if new_pool_rows:
        pool_collection = pd.concat([pool_collection, pd.DataFrame(new_pool_rows)], ignore_index=True)
    if new_reject_rows:
        rejected_pool = pd.concat([rejected_pool, pd.DataFrame(new_reject_rows)], ignore_index=True)

    current_index += 1
    if current_index >= len(pool_collection):
        current_index = 0
        current_hop += 1

# ====== END REPORT ======
end_time = time.time()
print(f"Total papers in pool_collection: {len(pool_collection)}")
print(f"Total papers in rejected_pool: {len(rejected_pool)}")
print(f"Execution time: {end_time - start_time:.2f} seconds")


Processing hop 1: W4394693583 - Acute Adverse Effects of Therapeutic Doses of Psilocybin


/opt/anaconda3/envs/TransformersCleanNLPEnv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


✅ New approved: 36 | ❌ Rejected: 22
Processing hop 1: W4410824431 - The Potential Role of Psilocybin in Traumatic Brain Injury Recovery: A Narrative Review


/opt/anaconda3/envs/TransformersCleanNLPEnv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


✅ New approved: 33 | ❌ Rejected: 22
Processing hop 1: W4308146982 - Single-Dose Psilocybin for a Treatment-Resistant Episode of Major Depression


/opt/anaconda3/envs/TransformersCleanNLPEnv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


✅ New approved: 88 | ❌ Rejected: 117
Processing hop 1: W2161555895 - Pilot Study of Psilocybin Treatment for Anxiety in Patients With Advanced-Stage Cancer


/opt/anaconda3/envs/TransformersCleanNLPEnv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


✅ New approved: 49 | ❌ Rejected: 144
Processing hop 1: W2559739670 - Rapid and sustained symptom reduction following psilocybin treatment for anxiety and depression in patients with life-threatening cancer: a randomized controlled trial


/opt/anaconda3/envs/TransformersCleanNLPEnv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


KeyboardInterrupt: 

In [59]:
pool_collection

,ID,Title,API link,Authors,Citation or Reference
0,W4394693583,Acute Adverse Effects of Therapeutic Doses of ...,https://api.openalex.org/works/W4394693583,"Akhila Yerubandi, Jennifer E. Thomas, N. M. Ma...",Seed


In [50]:
rejected_pool

,ID,Title,API link,Authors,Citation or Reference
0,W2118310760,Summary of the Second Report of the National C...,https://api.openalex.org/works/W2118310760,Scott M. Grundy,reference_&_citations
1,W4238251256,Executive Summary of the Third Report of the N...,https://api.openalex.org/works/W4238251256,Evaluation Expert Panel on Detection,reference_&_citations
2,W4285719527,Deleted Work,https://api.openalex.org/works/W4285719527,,reference_&_citations
3,W2779036087,Burnout: Moving beyond the status quo.,https://api.openalex.org/works/W2779036087,"Renzo Bianchi, Irvin Sam Schonfeld, Éric Laurent",reference_&_citations


In [51]:
print(f"Total papers in pool_collection : {len(pool_collection)}")

Total papers in pool_collection : 266


In [35]:
create_openalex_entry_df('W2148132316')

,ID,Title,API link,Citation or Reference,Authors
0,W2148132316,Evidence for a differential role of HPA-axis f...,https://api.openalex.org/works/W2148132316,Citation,"Femke Lamers, Nicole Vogelzangs, Kathleen R. M..."


In [69]:
pool_collection

,ID,Title,API link,Authors,Citation or Reference
0,W2148132316,Evidence for a differential role of HPA-axis f...,https://api.openalex.org/works/W2148132316,"Femke Lamers, Nicole Vogelzangs, Kathleen R. M...",Seed
